# Simulación · Unidad Modular III
## Variables aleatorias, bondad de ajuste, generación de variables y modelamiento

Cuaderno práctico que acompaña la presentación. Reproduce los ejemplos con **NumPy**, **pandas**, **SciPy**, **Matplotlib** y **seaborn**.

| Sección | Tema |
|---|---|
| 1 | Tipos de variable aleatoria y distribuciones |
| 2 | Formas de distribución de un conjunto de datos |
| 3 | Pruebas de bondad: chi-cuadrada, Kolmogorov-Smirnov y Anderson-Darling |
| 4 | Ajuste automático de distribuciones (al estilo de Stat::Fit) |
| 5 | Generación de variables: inversa, convolución, aceptación y rechazo, composición |
| 6 | Aplicaciones: líneas de espera, inventario y producción |
| 7 | Ejercicios y comprobación |

Ejecuta las celdas en orden (`Shift + Enter`) o usa *Entorno de ejecución → Ejecutar todas*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

MORADO, VERDE, AMBAR, CORAL, AZUL, TINTA = "#6C4AB6", "#00A896", "#F2A900", "#E4572E", "#3A86C8", "#1A1433"
PAL = [MORADO, VERDE, CORAL, AZUL, AMBAR]
sns.set_theme(style="whitegrid", palette=PAL)
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.precision", 4)

### Funciones de la unidad
Pruebas de bondad, métodos de generación y modelos. Se usan en todo el cuaderno.

In [ ]:
# ---------- pruebas de bondad ----------
def chi2_discreta(datos, pmf, valores, p_estimados=0, alfa=0.05, emin=5):
    """Chi-cuadrada para datos discretos. La última clase de `valores` acumula la cola (x ≥ último).
    Las clases de los extremos con E < emin se agrupan con su vecina."""
    datos = np.asarray(datos); n = len(datos)
    cl = []
    for k, x in enumerate(valores):
        cola = k == len(valores) - 1
        O = int(np.sum(datos >= x)) if cola else int(np.sum(datos == x))
        p = (1 - sum(c["p"] for c in cl)) if cola else pmf(x)
        cl.append(dict(lo=x, hi=None if cola else x, O=O, p=p))
    while len(cl) > 2 and n * cl[0]["p"] < emin:
        a = cl.pop(0); cl[0] = dict(lo=a["lo"], hi=cl[0]["hi"], O=a["O"] + cl[0]["O"], p=a["p"] + cl[0]["p"])
    while len(cl) > 2 and n * cl[-1]["p"] < emin:
        a = cl.pop(); cl[-1] = dict(lo=cl[-1]["lo"], hi=a["hi"], O=a["O"] + cl[-1]["O"], p=a["p"] + cl[-1]["p"])
    for c in cl:
        c["x"] = f"≥ {c['lo']}" if c["hi"] is None else (str(c["lo"]) if c["lo"] == c["hi"] else f"{c['lo']}–{c['hi']}")
        c["E"] = n * c["p"]; c["contrib"] = (c["O"] - c["E"]) ** 2 / c["E"]
    chi0 = sum(c["contrib"] for c in cl); gl = len(cl) - 1 - p_estimados
    crit = stats.chi2.ppf(1 - alfa, gl)
    return dict(n=n, clases=cl, chi0=chi0, gl=gl, crit=crit, p=1 - stats.chi2.cdf(chi0, gl), acepta=bool(chi0 <= crit))

def ks(datos, cdf, alfa=0.05):
    x = np.sort(np.asarray(datos)); n = len(x); F = cdf(x); i = np.arange(1, n + 1)
    dmas = i / n - F; dmenos = F - (i - 1) / n
    D = max(dmas.max(), dmenos.max()); crit = stats.kstwo.ppf(1 - alfa, n)
    return dict(n=n, x=x.tolist(), F=F.tolist(), dmas=dmas.tolist(), dmenos=dmenos.tolist(), Dmas=dmas.max(), Dmenos=dmenos.max(), D=D, crit=crit, acepta=bool(D <= crit))

AD_CRIT = {0.10: 1.933, 0.05: 2.492, 0.01: 3.857}  # distribución completamente especificada (caso 0)

def anderson_darling(datos, cdf, alfa=0.05):
    x = np.sort(np.asarray(datos)); n = len(x); F = np.clip(cdf(x), 1e-12, 1 - 1e-12); i = np.arange(1, n + 1)
    S = np.sum((2 * i - 1) * (np.log(F) + np.log(1 - F[::-1])))
    A2 = -n - S / n
    return dict(n=n, A2=A2, crit=AD_CRIT[alfa], acepta=bool(A2 <= AD_CRIT[alfa]), terminos=((2 * i - 1) * (np.log(F) + np.log(1 - F[::-1]))).tolist(), F=F.tolist(), x=x.tolist())

# ---------- generación ----------
def inv_exponencial(r, media): return -media * np.log(1 - np.asarray(r))
def inv_triangular(r, a, c, b):
    r = np.asarray(r, dtype=float); Fc = (c - a) / (b - a)
    return np.where(r < Fc, a + np.sqrt(r * (b - a) * (c - a)), b - np.sqrt((1 - r) * (b - a) * (b - c)))
def inv_discreta(r, valores, probs):
    acum = np.cumsum(probs); return [valores[int(np.searchsorted(acum, u, side="right"))] if u < acum[-1] else valores[-1] for u in np.atleast_1d(r)]
def erlang_convolucion(rs, k, media):
    """rs: arreglo (n, k). X = -(media/k) ln(prod r)."""
    rs = np.asarray(rs); return -(media / k) * np.log(np.prod(rs, axis=1))
def normal_suma12(rs, mu, sigma):
    return mu + sigma * (np.sum(rs, axis=1) - 6)
def aceptacion_rechazo(pares, f, a, b, M):
    out = []
    for r1, r2 in pares:
        x = a + (b - a) * r1; fx = f(x); acepta = r2 <= fx / M
        out.append(dict(r1=r1, r2=r2, x=x, fx=fx, umbral=fx / M, acepta=bool(acepta)))
    return out
def composicion(pares, pesos, medias):
    out = []; acum = np.cumsum(pesos)
    for r1, r2 in pares:
        j = int(np.searchsorted(acum, r1, side="right")); x = -medias[j] * np.log(1 - r2)
        out.append(dict(r1=r1, comp=j + 1, r2=r2, x=x))
    return out

# ---------- modelos ----------
def mm1_teoria(lam, mu):
    rho = lam / mu
    return dict(rho=rho, L=rho / (1 - rho), Lq=rho ** 2 / (1 - rho), W=1 / (mu - lam), Wq=lam / (mu * (mu - lam)), P0=1 - rho)

def simular_cola(n, lam, mu, rng, c=1):
    """Cola FIFO con c servidores, llegadas Poisson y servicio exponencial (tiempos en horas)."""
    ia = rng.exponential(1 / lam, n); sv = rng.exponential(1 / mu, n)
    llegada = np.cumsum(ia); libre = np.zeros(c); ini = np.empty(n); fin = np.empty(n)
    for i in range(n):
        j = int(np.argmin(libre)); ini[i] = max(llegada[i], libre[j]); fin[i] = ini[i] + sv[i]; libre[j] = fin[i]
    T = fin.max(); espera = ini - llegada
    return dict(llegada=llegada, ini=ini, fin=fin, servicio=sv, Wq=espera.mean(), W=(fin - llegada).mean(),
                util=sv.sum() / (c * T), Lq=espera.sum() / T, L=(fin - llegada).sum() / T)

def simular_inventario(dias, Q, R, inv0, dem_vals, dem_p, lt_vals, lt_p, K, h, s_costo, rng, detalle=False):
    inv = inv0; pedido = None; filas = []; costo = dict(pedido=0.0, mantener=0.0, faltante=0.0)
    for d in range(1, dias + 1):
        llega = 0
        if pedido and pedido[0] == d:
            inv += Q; llega = Q; pedido = None
        r = rng.random(); dem = inv_discreta([r], dem_vals, dem_p)[0]
        vend = min(inv, dem); falt = dem - vend; inv -= vend
        orden = ""
        if inv <= R and pedido is None:
            r2 = rng.random(); lt = inv_discreta([r2], lt_vals, lt_p)[0]
            pedido = (d + lt + 1, Q); costo["pedido"] += K; orden = f"Q={Q}, llega día {d + lt + 1}"
        costo["mantener"] += h * inv; costo["faltante"] += s_costo * falt
        if detalle:
            filas.append(dict(dia=d, recibe=llega, r=r, demanda=dem, vende=vend, faltante=falt, inv_final=inv, orden=orden))
    costo["total"] = sum(costo.values())
    return (filas, costo) if detalle else costo

def tiempo_pedido(rs, etapas):
    """rs: (n, k); etapas: lista de (a, c, b)."""
    return sum(inv_triangular(rs[:, j], *e) for j, e in enumerate(etapas))

---
## 1. Tipos de variable aleatoria

* **Discreta**: valores contables. Se describe con $p(x) = P(X = x)$.
* **Continua**: cualquier valor en un intervalo. Se describe con la densidad $f(x)$, y $P(a \le X \le b) = \int_a^b f(x)\,dx$.
* Ambas tienen acumulada $F(x) = P(X \le x)$.

In [ ]:
x = np.arange(0, 13)
discretas = pd.DataFrame({"x": x, "Poisson (λ=3)": stats.poisson.pmf(x, 3), "Binomial (n=10, p=0.3)": stats.binom.pmf(x, 10, 0.3),
                          "Geométrica (p=0.3)": stats.geom.pmf(x + 1, 0.3)})
largo = discretas.melt("x", var_name="distribución", value_name="p(x)")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=largo, x="x", y="p(x)", hue="distribución", ax=ax[0]); ax[0].set_title("Funciones de probabilidad (discretas)")
t = np.linspace(0, 12, 400)
for (nombre, dist), c in zip({"Exponencial (media 2)": stats.expon(scale=2), "Normal (5; 1.2)": stats.norm(5, 1.2), "Gamma (k=3; θ=1)": stats.gamma(3, scale=1),
                               "Weibull (k=2; λ=5)": stats.weibull_min(2, scale=5), "Triangular (1; 3; 9)": stats.triang((3 - 1) / 8, loc=1, scale=8)}.items(), PAL):
    ax[1].plot(t, dist.pdf(t), lw=2.2, color=c, label=nombre)
ax[1].set_title("Funciones de densidad (continuas)"); ax[1].legend(); plt.tight_layout(); plt.show()

Tabla de media y varianza teóricas, comparadas con una muestra de 10 000 valores generados por SciPy.

In [ ]:
rng = np.random.default_rng(1)
dists = {"Poisson (λ=3)": stats.poisson(3), "Binomial (10; 0.3)": stats.binom(10, 0.3), "Exponencial (media 2)": stats.expon(scale=2),
         "Normal (5; 1.2)": stats.norm(5, 1.2), "Gamma (3; 1)": stats.gamma(3), "Weibull (2; 5)": stats.weibull_min(2, scale=5)}
filas = []
for n, d in dists.items():
    m = d.rvs(10000, random_state=rng)
    filas.append({"distribución": n, "tipo": "discreta" if hasattr(d.dist, "pmf") else "continua", "media teórica": d.mean(), "media muestral": m.mean(),
                  "varianza teórica": d.var(), "varianza muestral": m.var(ddof=1)})
pd.DataFrame(filas).set_index("distribución")

---
## 2. Formas de distribución de un conjunto de datos

La forma del histograma y la relación entre media y mediana orientan la elección de la distribución:
media > mediana → sesgo a la derecha; media < mediana → sesgo a la izquierda; dos picos → mezcla.

In [ ]:
rng = np.random.default_rng(12)
formas = {"Simétrica": rng.normal(50, 10, 500), "Sesgada a la derecha": 20 + rng.gamma(2, 10, 500), "Sesgada a la izquierda": 100 - rng.gamma(2, 10, 500),
          "Uniforme": rng.uniform(20, 80, 500), "Bimodal": np.r_[rng.normal(35, 6, 250), rng.normal(68, 6, 250)]}
fig, axs = plt.subplots(1, 5, figsize=(18, 3.6), sharey=True)
resumen = []
for (n, v), a, c in zip(formas.items(), axs, PAL):
    sns.histplot(v, bins=20, ax=a, color=c); a.axvline(v.mean(), color=TINTA, lw=2, label="media"); a.axvline(np.median(v), color=AMBAR, lw=2, ls="--", label="mediana")
    a.set_title(n, fontsize=11)
    resumen.append({"forma": n, "media": v.mean(), "mediana": np.median(v), "desv.": v.std(ddof=1), "CV": v.std(ddof=1) / v.mean(), "asimetría": stats.skew(v), "curtosis": stats.kurtosis(v)})
axs[0].legend(); plt.tight_layout(); plt.show()
pd.DataFrame(resumen).set_index("forma")

---
## 3. Pruebas de bondad de ajuste

### 3.1 Chi-cuadrada: llegadas por intervalo (¿Poisson?)
$$\chi^2_0 = \sum \frac{(O_i - E_i)^2}{E_i}, \qquad gl = k - 1 - p$$

In [ ]:
llegadas = np.random.default_rng(2026).poisson(3, 100)        # los mismos datos de la presentación
lam = llegadas.mean()
res = chi2_discreta(llegadas, lambda x: stats.poisson.pmf(x, lam), list(range(0, 9)), p_estimados=1)
tabla = pd.DataFrame(res["clases"])[["x", "O", "p", "E", "contrib"]].set_index("x")
display(tabla)
print(f"λ estimado = {lam:.2f}   χ²₀ = {res['chi0']:.3f}   gl = {res['gl']}   crítico = {res['crit']:.3f}   p = {res['p']:.3f}")
print("Decisión:", "no se rechaza H0" if res["acepta"] else "se rechaza H0")
print("Comprobación con SciPy:", stats.chisquare(tabla["O"], tabla["E"], ddof=1))
tabla[["O", "E"]].plot.bar(color=[MORADO, AMBAR], rot=0); plt.title("Observada frente a esperada"); plt.show()

### 3.2 Kolmogorov-Smirnov y Anderson-Darling: 20 tiempos de servicio (¿exponencial de media 4?)
$$D = \max\left(\max_i\left(\tfrac{i}{n} - F_0(x_{(i)})\right),\ \max_i\left(F_0(x_{(i)}) - \tfrac{i-1}{n}\right)\right)$$
$$A^2 = -n - \frac1n\sum_{i=1}^{n}(2i-1)\left[\ln F_0(x_{(i)}) + \ln\left(1 - F_0(x_{(n+1-i)})\right)\right]$$

In [ ]:
servicio = np.round(np.random.default_rng(31).exponential(4, 20), 2)
F0 = lambda x: stats.expon.cdf(x, scale=4)
k, a = ks(servicio, F0), anderson_darling(servicio, F0)
display(pd.DataFrame({"x(i)": k["x"], "F0": k["F"], "D+": k["dmas"], "D-": k["dmenos"], "término A-D": a["terminos"]}, index=pd.RangeIndex(1, 21, name="i")).T)
print(f"K-S: D = {k['D']:.4f}  crítico = {k['crit']:.3f}  → {'no se rechaza' if k['acepta'] else 'se rechaza'}")
print(f"A-D: A² = {a['A2']:.3f}  crítico = {a['crit']}  → {'no se rechaza' if a['acepta'] else 'se rechaza'}")
print("SciPy kstest:", stats.kstest(servicio, "expon", args=(0, 4)))
gof = stats.goodness_of_fit(stats.expon, servicio, known_params={"loc": 0, "scale": 4}, statistic="ad", random_state=1)
print(f"SciPy goodness_of_fit (A-D): estadístico = {gof.statistic:.3f}, valor p = {gof.pvalue:.3f}")

xs = np.sort(servicio); t = np.linspace(0, 14, 300)
plt.step(np.r_[0, xs, 14], np.r_[0, np.arange(1, 21) / 20, 1], where="post", color=MORADO, lw=2, label="acumulada empírica")
plt.plot(t, F0(t), color=CORAL, lw=2, label="exponencial media 4")
i = int(np.argmax(np.maximum(k["dmas"], k["dmenos"])))
plt.vlines(xs[i], F0(xs[i]), (i + 1) / 20, color=AMBAR, lw=3, label=f"D = {k['D']:.3f}")
plt.title("Kolmogorov-Smirnov: la mayor separación vertical"); plt.legend(); plt.show()

**¿Y si los parámetros se estiman con los mismos datos?** Las tablas clásicas de K-S suponen $F_0$ totalmente conocida. Con la media estimada, el estadístico tiende a ser menor y la prueba se vuelve permisiva. Este experimento lo muestra con 2 000 muestras exponenciales: la proporción de rechazos queda muy por debajo del 5 %.

In [ ]:
rng = np.random.default_rng(3)
rech_fijo, rech_est = [], []
for _ in range(2000):
    x = rng.exponential(4, 20)
    rech_fijo.append(stats.kstest(x, "expon", args=(0, 4)).pvalue < 0.05)
    rech_est.append(stats.kstest(x, "expon", args=(0, x.mean())).pvalue < 0.05)
print(f"Rechazos con la media conocida (4):        {np.mean(rech_fijo):.3f}  (se espera 0.05)")
print(f"Rechazos con la media estimada de los datos: {np.mean(rech_est):.3f}  (la tabla clásica ya no es válida → Lilliefors)")

---
## 4. Ajuste automático de distribuciones (al estilo de Stat::Fit)

Stat::Fit estima los parámetros de varias distribuciones candidatas, aplica las pruebas de bondad y las ordena. La función `autoajuste` hace lo mismo con SciPy:
estimación por máxima verosimilitud, K-S, A-D, chi-cuadrada con 10 clases equiprobables y criterio AIC.

In [ ]:
CANDIDATAS = {"Exponencial": (stats.expon, {"floc": 0}), "Gamma": (stats.gamma, {"floc": 0}), "Lognormal": (stats.lognorm, {"floc": 0}),
              "Weibull": (stats.weibull_min, {"floc": 0}), "Normal": (stats.norm, {})}

def autoajuste(datos, candidatas=CANDIDATAS, clases=10):
    datos = np.asarray(datos); n = len(datos); filas = []
    for nombre, (dist, kw) in candidatas.items():
        par = dist.fit(datos, **kw); cdf = lambda x, d=dist, p=par: d.cdf(x, *p)
        k = ks(datos, cdf); a = anderson_darling(datos, cdf)
        bordes = dist.ppf(np.linspace(0, 1, clases + 1), *par); O = np.histogram(datos, bins=bordes)[0]
        npar = len(par) - (1 if "floc" in kw else 0)
        chi0 = ((O - n / clases) ** 2 / (n / clases)).sum()
        filas.append({"distribución": nombre, "parámetros": tuple(np.round(par, 3)), "K-S D": k["D"], "A-D A²": a["A2"],
                      "χ²₀": chi0, "p (χ²)": 1 - stats.chi2.cdf(chi0, clases - 1 - npar), "AIC": 2 * npar - 2 * np.sum(dist.logpdf(datos, *par))})
    return pd.DataFrame(filas).sort_values("A-D A²").reset_index(drop=True).rename_axis("rango").rename(index=lambda i: i + 1)

datos = np.round(np.random.default_rng(7).gamma(2.5, 2.0, 200), 2)   # 200 tiempos de servicio de la presentación
print(pd.Series(datos).describe().round(3).to_string())
print(f"CV = {datos.std(ddof=1) / datos.mean():.3f}   asimetría = {stats.skew(datos, bias=False):.3f}")
ranking = autoajuste(datos); ranking

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
sns.histplot(datos, bins=20, stat="density", color="#CAC0E6", ax=ax[0])
t = np.linspace(0.01, datos.max(), 300)
for (nombre, (dist, kw)), c in zip(CANDIDATAS.items(), PAL):
    par = dist.fit(datos, **kw); ax[0].plot(t, dist.pdf(t, *par), lw=2, color=c, label=nombre)
ax[0].set_title("Histograma y densidades ajustadas"); ax[0].legend()
# Gráfica P-P de las dos mejores
xs = np.sort(datos); emp = (np.arange(1, len(xs) + 1) - 0.5) / len(xs)
for nombre, c in [(ranking.iloc[0, 0], VERDE), ("Exponencial", CORAL)]:
    dist, kw = CANDIDATAS[nombre]; par = dist.fit(datos, **kw)
    ax[1].plot(emp, dist.cdf(xs, *par), "o", ms=3, color=c, label=nombre)
ax[1].plot([0, 1], [0, 1], color=TINTA, lw=1); ax[1].set(title="Gráfica P-P: cerca de la diagonal = buen ajuste", xlabel="Probabilidad empírica", ylabel="Probabilidad teórica"); ax[1].legend()
plt.tight_layout(); plt.show()

Exportar la distribución elegida a un modelo de simulación es tan simple como escribir sus parámetros, por ejemplo `Gamma(k = 2.65, θ = 1.70)`. En Python se usa directamente:

In [ ]:
mejor, (dist, kw) = ranking.iloc[0, 0], CANDIDATAS[ranking.iloc[0, 0]]
par = dist.fit(datos, **kw)
nuevos = dist.rvs(*par, size=5, random_state=np.random.default_rng(0))
print(f"Distribución elegida: {mejor} {np.round(par, 3)}")
print("Cinco tiempos de servicio generados para el modelo:", np.round(nuevos, 2))

---
## 5. Generación de variables aleatorias

### 5.1 Transformada inversa
$$r = F(x) \;\Rightarrow\; x = F^{-1}(r)$$

In [ ]:
R = np.array([0.2311, 0.6068, 0.4860, 0.8913, 0.7621, 0.4565])
pd.DataFrame({"r": R, "Exponencial media 4": inv_exponencial(R, 4), "Triangular (2, 3, 5)": inv_triangular(R, 2, 3, 5),
              "Demanda discreta": inv_discreta(R, [0, 1, 2, 3, 4], [0.10, 0.25, 0.35, 0.20, 0.10])})

In [ ]:
rng = np.random.default_rng(10); r = rng.random(20000)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(inv_exponencial(r, 4), stat="density", bins=60, ax=ax[0], color=MORADO); t = np.linspace(0, 30, 300); ax[0].plot(t, stats.expon.pdf(t, scale=4), color=AMBAR, lw=2.5); ax[0].set_title("Exponencial media 4"); ax[0].set_xlim(0, 30)
sns.histplot(inv_triangular(r, 2, 3, 5), stat="density", bins=60, ax=ax[1], color=VERDE); t = np.linspace(2, 5, 200); ax[1].plot(t, stats.triang.pdf(t, 1 / 3, loc=2, scale=3), color=AMBAR, lw=2.5); ax[1].set_title("Triangular (2, 3, 5)")
dem = pd.Series(inv_discreta(r[:5000], [0, 1, 2, 3, 4], [0.10, 0.25, 0.35, 0.20, 0.10])).value_counts(normalize=True).sort_index()
ax[2].bar(dem.index - 0.2, dem.values, 0.4, color=CORAL, label="generada"); ax[2].bar(np.arange(5) + 0.2, [0.10, 0.25, 0.35, 0.20, 0.10], 0.4, color=AMBAR, label="teórica"); ax[2].legend(); ax[2].set_title("Discreta por tabla")
plt.tight_layout(); plt.show()

### 5.2 Convolución
Erlang $(k, \text{media } \theta)$: $X = -\frac{\theta}{k}\ln(r_1 r_2 \cdots r_k)$. Normal aproximada: $Z = \sum_{i=1}^{12} r_i - 6$.

In [ ]:
R3 = np.array([[0.2311, 0.6068, 0.4860], [0.8913, 0.7621, 0.4565], [0.0185, 0.8214, 0.4447], [0.6154, 0.7919, 0.9218]])
display(pd.DataFrame({"r1": R3[:, 0], "r2": R3[:, 1], "r3": R3[:, 2], "producto": R3.prod(1), "x (Erlang k=3, media 6)": erlang_convolucion(R3, 3, 6)}))
rng = np.random.default_rng(99)
erl = erlang_convolucion(rng.random((20000, 3)), 3, 6); nor = normal_suma12(rng.random((20000, 12)), 10, 2)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(erl, stat="density", bins=60, ax=ax[0], color=MORADO); t = np.linspace(0, 25, 300); ax[0].plot(t, stats.gamma.pdf(t, 3, scale=2), color=AMBAR, lw=2.5); ax[0].set_title("Erlang k = 3 por convolución")
sns.histplot(nor, stat="density", bins=60, ax=ax[1], color=VERDE); t = np.linspace(2, 18, 300); ax[1].plot(t, stats.norm.pdf(t, 10, 2), color=AMBAR, lw=2.5); ax[1].set_title("Normal (10, 2) como suma de 12 uniformes")
plt.tight_layout(); plt.show()

### 5.3 Aceptación y rechazo
$f(x) = 6x(1 - x)$ en $[0, 1]$ con $M = 1.5$: se acepta $x = r_1$ si $r_2 \le f(x)/M$. Eficiencia teórica $= 1/(M(b - a)) = 2/3$.

In [ ]:
pares = [(0.2311, 0.6068), (0.4860, 0.8913), (0.7621, 0.4565), (0.0185, 0.8214), (0.4447, 0.6154), (0.7919, 0.9218)]
display(pd.DataFrame(aceptacion_rechazo(pares, lambda x: 6 * x * (1 - x), 0, 1, 1.5)))
rng = np.random.default_rng(5); p = rng.random((3000, 2)); x, y = p[:, 0], 1.5 * p[:, 1]; acc = y <= 6 * x * (1 - x)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].scatter(x[acc], y[acc], s=4, color=VERDE, label="aceptados"); ax[0].scatter(x[~acc], y[~acc], s=4, color=CORAL, label="rechazados")
t = np.linspace(0, 1, 200); ax[0].plot(t, 6 * t * (1 - t), color=TINTA, lw=2); ax[0].set_title(f"Eficiencia observada: {acc.mean():.3f} (teórica 0.667)"); ax[0].legend()
sns.histplot(x[acc], stat="density", bins=40, ax=ax[1], color=MORADO); ax[1].plot(t, 6 * t * (1 - t), color=AMBAR, lw=2.5); ax[1].set_title("Los aceptados siguen f(x)")
plt.tight_layout(); plt.show()

### 5.4 Composición
Servicio: 70 % sencillo (exponencial media 2) y 30 % complejo (exponencial media 8). Media = 0.7·2 + 0.3·8 = 3.8 min.

In [ ]:
display(pd.DataFrame(composicion(pares, [0.7, 0.3], [2, 8])))
rng = np.random.default_rng(6); n = 20000
mezcla = np.where(rng.random(n) < 0.7, rng.exponential(2, n), rng.exponential(8, n))
print(f"Media simulada = {mezcla.mean():.3f} (teórica 3.8);  CV = {mezcla.std() / mezcla.mean():.3f} (> 1: hiperexponencial)")
t = np.linspace(0, 40, 400)
sns.histplot(mezcla, stat="density", bins=100, color=AZUL); plt.plot(t, 0.7 * stats.expon.pdf(t, scale=2) + 0.3 * stats.expon.pdf(t, scale=8), color=AMBAR, lw=2.5)
plt.xlim(0, 40); plt.title("Composición: mezcla de dos exponenciales"); plt.show()

### 5.5 ¿Qué tan rápido es cada método?

In [ ]:
import time
n, rng = 200_000, np.random.default_rng(0)
def tiempo(f):
    t0 = time.perf_counter(); f(); return (time.perf_counter() - t0) * 1000
pd.Series({"Inversa (exponencial)": tiempo(lambda: inv_exponencial(rng.random(n), 4)),
           "Convolución (Erlang k=3)": tiempo(lambda: erlang_convolucion(rng.random((n, 3)), 3, 6)),
           "Aceptación-rechazo (beta)": tiempo(lambda: (lambda p: p[1.5 * p[:, 1] <= 6 * p[:, 0] * (1 - p[:, 0]), 0])(rng.random((n * 3 // 2, 2)))),
           "Composición (mezcla)": tiempo(lambda: np.where(rng.random(n) < 0.7, rng.exponential(2, n), rng.exponential(8, n)))}, name="milisegundos para 200 000 valores").round(1).to_frame()

---
## 6. Aplicaciones

### 6.1 Líneas de espera M/M/1 (y M/M/2)
$\lambda = 10$ clientes/h, $\mu = 12$ clientes/h.

In [ ]:
teo = mm1_teoria(10, 12)
reps = pd.DataFrame([simular_cola(5000, 10, 12, np.random.default_rng(100 + k)) for k in range(30)])[["util", "L", "Lq", "W", "Wq"]]
comp = pd.DataFrame({"teoría": [teo["rho"], teo["L"], teo["Lq"], teo["W"] * 60, teo["Wq"] * 60],
                     "simulación (media)": [reps.util.mean(), reps.L.mean(), reps.Lq.mean(), reps.W.mean() * 60, reps.Wq.mean() * 60],
                     "desv. entre réplicas": [reps.util.std(), reps.L.std(), reps.Lq.std(), reps.W.std() * 60, reps.Wq.std() * 60]},
                    index=["ρ", "L", "Lq", "W (min)", "Wq (min)"])
comp

In [ ]:
# ¿Qué pasa al aumentar la carga? y ¿cuánto ayuda un segundo cajero?
filas = []
for lam in [4, 6, 8, 9, 10, 11, 11.5]:
    for c in (1, 2):
        s = [simular_cola(4000, lam, 12, np.random.default_rng(int(lam * 10) + k + 500 * c), c=c)["Wq"] * 60 for k in range(10)]
        filas.append({"λ (clientes/h)": lam, "cajeros": c, "Wq simulada (min)": np.mean(s)})
carga = pd.DataFrame(filas)
sns.lineplot(data=carga, x="λ (clientes/h)", y="Wq simulada (min)", hue="cajeros", marker="o", palette=[CORAL, VERDE])
lam = np.linspace(4, 11.5, 100); plt.plot(lam, 60 * lam / (12 * (12 - lam)), color=TINTA, ls="--", label="teoría M/M/1")
plt.title("Espera en cola según la tasa de llegadas (μ = 12/h)"); plt.legend(); plt.show()

### 6.2 Inventario con política (Q, R)
Demanda diaria 6, 8, 10, 12, 14 (p = 0.10, 0.25, 0.35, 0.20, 0.10); entrega en 1–3 días (p = 0.3, 0.5, 0.2); K = 100, h = 1, faltante = 5.

In [ ]:
dem_v, dem_p, lt_v, lt_p = [6, 8, 10, 12, 14], [0.10, 0.25, 0.35, 0.20, 0.10], [1, 2, 3], [0.3, 0.5, 0.2]
filas, costo = simular_inventario(30, 60, 25, 50, dem_v, dem_p, lt_v, lt_p, 100, 1, 5, np.random.default_rng(5), detalle=True)
inv = pd.DataFrame(filas).set_index("dia")
display(inv.head(12)); print("Costos de 30 días:", costo)
plt.step(inv.index, inv["inv_final"], where="mid", color=MORADO, lw=2, label="inventario final")
plt.axhline(25, color=CORAL, ls="--", label="R = 25"); plt.bar(inv.index, inv["faltante"], color=AMBAR, label="faltante")
plt.title("Inventario durante 30 días"); plt.xlabel("día"); plt.legend(); plt.show()

In [ ]:
# Búsqueda de la mejor política: costo promedio por día (30 réplicas de 360 días)
Qs, Rs = [30, 40, 50, 60, 80, 100], [15, 20, 25, 30, 35]
mapa = pd.DataFrame(index=pd.Index(Rs, name="R"), columns=pd.Index(Qs, name="Q"), dtype=float)
for Q in Qs:
    for Rr in Rs:
        mapa.loc[Rr, Q] = np.mean([simular_inventario(360, Q, Rr, 50, dem_v, dem_p, lt_v, lt_p, 100, 1, 5, np.random.default_rng(7000 + k))["total"] / 360 for k in range(30)])
sns.heatmap(mapa, annot=True, fmt=".1f", cmap="Purples_r", cbar_kws={"label": "costo por día ($)"})
mejor = mapa.stack().idxmin(); plt.title(f"Costo por día según (Q, R) · mejor: R = {mejor[0]}, Q = {mejor[1]}"); plt.show()

### 6.3 Producción: tiempo total de un pedido
Corte Tri(2, 3, 5) h + ensamble Tri(4, 6, 9) h + pintura Tri(1, 2, 4) h.

In [ ]:
etapas = [(2, 3, 5), (4, 6, 9), (1, 2, 4)]
T = tiempo_pedido(np.random.default_rng(8).random((10000, 3)), etapas)
print(f"Media = {T.mean():.2f} h (teórica {sum((a + b + c) / 3 for a, c, b in etapas):.2f})   P(T ≤ 12) = {np.mean(T <= 12):.3f}   P(T ≤ 14) = {np.mean(T <= 14):.3f}   P90 = {np.percentile(T, 90):.2f} h")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(T, bins=50, ax=ax[0], color=AZUL); ax[0].axvline(12, color=CORAL, ls="--", label="plazo 12 h"); ax[0].legend(); ax[0].set_title("Tiempo total de 10 000 pedidos")
plazos = np.linspace(9, 16, 50); ax[1].plot(plazos, [np.mean(T <= p) for p in plazos], color=VERDE, lw=2.5)
ax[1].axhline(0.95, color=CORAL, ls="--"); ax[1].set(title="Probabilidad de cumplir según el plazo prometido", xlabel="plazo (h)", ylabel="P(T ≤ plazo)")
plt.tight_layout(); plt.show()
print(f"Plazo para cumplir el 95 % de los pedidos: {np.percentile(T, 95):.2f} h")

---
## 7. Ejercicios

1. Clasifica como discreta o continua: clientes en cola, tiempo de servicio, defectuosos en un lote, temperatura de un horno.
2. **Chi-cuadrada.** 60 días con demanda 0, 1, 2, 3 observada 10, 22, 18, 10 veces. ¿Ajusta p = 0.2, 0.3, 0.3, 0.2?
3. **K-S.** ¿Los datos 0.8, 1.9, 2.6, 4.4, 7.1 siguen una exponencial de media 3?
4. **M/M/1.** λ = 8/h, μ = 10/h: ρ, L, Lq, W, Wq.
5. Media 18 y mediana 12: ¿qué forma tienen los datos?
6. x̄ = 5 y S = 4.9: ¿qué sugiere el CV?
7. **Inversa exponencial** media 5 con r = 0.25 y 0.70.
8. **Inversa triangular** Tri(2, 4, 8) con r = 0.2 y 0.8.
9. **Inversa discreta** (tabla de demanda) con r = 0.05, 0.40, 0.95.
10. **Convolución** Erlang k = 2, media 4 con (0.3, 0.6) y (0.8, 0.5).
11. **Aceptación y rechazo** f(x) = 2x, M = 2, pares (0.6, 0.5) y (0.3, 0.7).
12. **Composición** 60 % exp. media 3 y 40 % media 10, r₁ = 0.75, r₂ = 0.5.

**Retos:** (a) agrega la distribución Erlang (gamma con k entero) a `CANDIDATAS`; (b) simula una cola M/M/1 con servicio gamma ajustado en la sección 4; (c) encuentra la política (Q, R) óptima si el faltante cuesta 20.

In [ ]:
O, E = np.array([10, 22, 18, 10]), 60 * np.array([0.2, 0.3, 0.3, 0.2])
k = ks([0.8, 1.9, 2.6, 4.4, 7.1], lambda x: stats.expon.cdf(x, scale=3)); t = mm1_teoria(8, 10)
print("1) discreta, continua, discreta, continua")
print(f"2) χ²₀ = {((O - E) ** 2 / E).sum():.3f}, crítico = {stats.chi2.ppf(0.95, 3):.3f}")
print(f"3) D+ = {k['Dmas']:.3f}, D- = {k['Dmenos']:.3f}, D = {k['D']:.3f}, crítico = {k['crit']:.3f}")
print(f"4) ρ = {t['rho']:.2f}, L = {t['L']:.2f}, Lq = {t['Lq']:.2f}, W = {t['W'] * 60:.1f} min, Wq = {t['Wq'] * 60:.1f} min")
print("5) sesgada a la derecha;  6) CV =", round(4.9 / 5, 2), "→ exponencial")
print("7)", np.round(inv_exponencial([0.25, 0.70], 5), 3))
print("8)", np.round(inv_triangular([0.2, 0.8], 2, 4, 8), 3))
print("9)", inv_discreta([0.05, 0.40, 0.95], [0, 1, 2, 3, 4], [0.10, 0.25, 0.35, 0.20, 0.10]))
print("10)", np.round(erlang_convolucion([[0.3, 0.6], [0.8, 0.5]], 2, 4), 3))
print("11)", [(e["x"], e["acepta"]) for e in aceptacion_rechazo([(0.6, 0.5), (0.3, 0.7)], lambda x: 2 * x, 0, 1, 2)], "eficiencia = 0.5")
print("12)", composicion([(0.75, 0.5)], [0.6, 0.4], [3, 10]), " media =", 0.6 * 3 + 0.4 * 10)